# EdgeTRM Consolidated Evaluation Notebook

This notebook consolidates the evaluation pipelines for all three EdgeTRM model variants:
1. **ARC-Prize 2024 (Baseline)**
2. **Maze (Hard Version)**
3. **Sudoku (Extreme Version)**

Experiments are evaluated across **3 random seeds (42, 43, 44)** for scientific stability and compared side-by-side in Pandas tables.

In [1]:
import os
os.chdir("/lambda/nfs/EdgeTRM")
print(os.getcwd())

!git config --global --add safe.directory /lambda/nfs/EdgeTRM
# !rm -rf /lambda/nfs/EdgeTRM/EdgeTRM
 


/lambda/nfs/EdgeTRM


In [2]:
# !rm -rf /lambda/nfs/EdgeTRM/data/arc2test-aug-128_2024
# !unzip data_ne.zip -d .

In [3]:
# !git fetch origin
# !git reset --hard origin/main
!git pull origin main

From https://github.com/Seqaeon/EdgeTRM
 * branch            main       -> FETCH_HEAD
Already up to date.


In [4]:
# ── MUST RUN FIRST — fix duplicate trm.py on Modal volume ────────────────────
# There are two identical trm.py files on the Modal volume:
#   TinyRecursiveModels/trm.py
#   TinyRecursiveModels/models/recursive_reasoning/trm.py
#
# Python loads them as separate module objects, so patching one class
# has no effect on instances created from the other.
#
# Fix: replace the top-level copy with a symlink so both import paths
# resolve to the same file and the same Python module object.
#
# Run this cell ONCE per kernel, then restart the kernel.

import os, sys

trm_root = None
for path in sys.path:
    candidate = os.path.join(path, "trm.py")
    if os.path.exists(candidate) and "TinyRecursiveModels" in candidate:
        trm_root = candidate
        break

# Also check the known Modal volume path directly
modal_top = "/lambda/nfs/EdgeTRM/TinyRecursiveModels/trm.py"
modal_sub = "/lambda/nfs/EdgeTRM/TinyRecursiveModels/models/recursive_reasoning/trm.py"

for top, sub in [(modal_top, modal_sub)]:
    if not os.path.exists(sub):
        print(f"[SKIP] {sub} not found")
        continue
    if os.path.islink(top):
        print(f"✓ Already a symlink: {top} → {os.readlink(top)}")
    elif os.path.exists(top):
        os.rename(top, top + ".bak")
        os.symlink(sub, top)
        print(f"✓ Replaced {top} with symlink → {sub}")
        print("  Restart the kernel now, then run all cells from the top.")
    else:
        print(f"[SKIP] {top} not found")


✓ Already a symlink: /lambda/nfs/EdgeTRM/TinyRecursiveModels/trm.py → models/recursive_reasoning/trm.py


In [5]:
from pathlib import Path
import sys

repo_root = Path.cwd()
trm_root = repo_root / "TinyRecursiveModels"
if str(trm_root) not in sys.path:
    sys.path.insert(0, str(trm_root))

print("repo_root:", repo_root)
print("trm_root:", trm_root)

repo_root: /lambda/nfs/EdgeTRM
trm_root: /lambda/nfs/EdgeTRM/TinyRecursiveModels


In [6]:
# 1. Install uv globally for the user session
!pip install --user uv
# 1. Create the virtual environment using uv (lightning fast)
# !uv venv .venv --

# 2. Force copy mode for your local package on the NFS drive
!uv pip install --link-mode=copy --python .venv/bin/python {trm_root}

# 3. Force copy mode for einops
!uv pip install --link-mode=copy --python .venv/bin/python einops


!uv pip install --link-mode=copy --python .venv/bin/python matplotlib pandas

# Install ipykernel into your virtual environment
!uv pip install --python .venv/bin/python ipykernel

# Register the virtual environment as a new Jupyter Kernel
! .venv/bin/python -m ipykernel install --user --name=trm-env --display-name="Python (TRM Env)"


Resolved 64 packages in 321ms                                        
Prepared 1 package in 784ms                                              
Uninstalled 1 package in 64ms
Installed 1 package in 235ms==0.1.0 (from file:///lambda/nfs
 ~ tiny-recursive-models==0.1.0 (from file:///lambda/nfs/EdgeTRM/TinyRecursiveModels)
Checked 1 package in 30ms
Checked 2 packages in 35ms
Checked 1 package in 59ms
Installed kernelspec trm-env in /home/ubuntu/.local/share/jupyter/kernels/trm-env


In [7]:
import sys

# Get the exact python binary your notebook is using right now
active_python = sys.executable
print(f"Targeting environment: {active_python}")

# Force uv to install into this exact environment path using copy mode
!uv pip install --link-mode=copy --python {active_python} matplotlib pandas einops

Targeting environment: /lambda/nfs/EdgeTRM/.venv/bin/python
Checked 3 packages in 38ms


In [8]:

!uv pip install {trm_root}
!uv pip install einops

Resolved 64 packages in 178ms                                        
Prepared 1 package in 742ms                                              
Uninstalled 1 package in 53ms
░░░░░░░░░░░░░░░░░░░░ [0/1] Installing wheels...                                 warning: Failed to hardlink files; falling back to full copy. This may lead to degraded performance.
         If the cache and target directories are on different filesystems, hardlinking may not be supported.
         If this is intentional, set `export UV_LINK_MODE=copy` or use `--link-mode=copy` to suppress this warning.
Installed 1 package in 253ms==0.1.0 (from file:///lambda/nfs
 ~ tiny-recursive-models==0.1.0 (from file:///lambda/nfs/EdgeTRM/TinyRecursiveModels)
Checked 1 package in 34ms


In [9]:
!pip install pydantic

Defaulting to user installation because normal site-packages is not writeable


In [10]:
# import os
# os.chdir('/root/EdgeTRM/TinyRecursiveModels')


## Section 1 — Setup and global imports

In [11]:
import sys, time, copy, json, math, warnings, io, os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import pandas as pd
warnings.filterwarnings("ignore")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

Device: cuda


## Section 2 — Model Configuration & Checkpoint Loading

In [20]:
import sys, os
trm_root = os.path.abspath("TinyRecursiveModels")
if trm_root not in sys.path:
    sys.path.insert(0, trm_root)

from trm import TinyRecursiveReasoningModel_ACTV1, TinyRecursiveReasoningModel_ACTV1Carry, TinyRecursiveReasoningModel_ACTV1InnerCarry
import yaml

# Flat ARC 2024 config directly satisfying TinyRecursiveReasoningModel_ACTV1Config validation rules
arc_config = {
    "batch_size": 32,
    "seq_len": 900,
    "num_puzzle_identifiers": 50911,
    "vocab_size": 12,
    "H_cycles": 3,
    "L_cycles": 4,
    "H_layers": 0,
    "L_layers": 2,
    "hidden_size": 512,
    "expansion": 4,
    "num_heads": 8,
    "pos_encodings": "rope",
    "halt_max_steps": 16,
    "halt_exploration_prob": 0.1,
    "forward_dtype": "bfloat16",
    "mlp_t": False,
    "puzzle_emb_ndim": 512,
    "puzzle_emb_len": 16,
    "no_ACT_continue": True
}

# Flat Maze Hard config
maze_config = {
    "batch_size": 32,
    "seq_len": 900,
    "num_puzzle_identifiers": 1,
    "vocab_size": 6,
    "H_cycles": 3,
    "L_cycles": 4,
    "H_layers": 0,
    "L_layers": 2,
    "hidden_size": 512,
    "expansion": 4,
    "num_heads": 8,
    "pos_encodings": "rope",
    "halt_max_steps": 16,
    "halt_exploration_prob": 0.1,
    "forward_dtype": "bfloat16",
    "mlp_t": False,
    "puzzle_emb_ndim": 512,
    "puzzle_emb_len": 16,
    "no_ACT_continue": True
}

# Flat Sudoku Extreme config
sudoku_config = {
    "batch_size": 32,
    "seq_len": 81,
    "num_puzzle_identifiers": 1,
    "vocab_size": 11,
    "H_cycles": 3,
    "L_cycles": 6,
    "H_layers": 0,
    "L_layers": 2,
    "hidden_size": 512,
    "expansion": 4,
    "num_heads": 8,
    "pos_encodings": "none",
    "halt_max_steps": 16,
    "halt_exploration_prob": 0.1,
    "forward_dtype": "bfloat16",
    "mlp_t": True,
    "puzzle_emb_ndim": 512,
    "puzzle_emb_len": 16,
    "no_ACT_continue": True
}

def get_inner(m):
    m2 = m.module if hasattr(m, 'module') else m
    return m2._orig_mod if hasattr(m2, '_orig_mod') else m2

def load_trm_model(name, checkpoint_path, config_dict):
    print(f"Loading {name} model from {checkpoint_path}...")
    model = TinyRecursiveReasoningModel_ACTV1(config_dict=config_dict)
    if not os.path.exists(checkpoint_path):
        print(f"[WARNING] Checkpoint {checkpoint_path} not found. Skipping weight loading.")
        return model
        
    state_dict = torch.load(checkpoint_path, map_location='cpu')
    if 'model' in state_dict:
        state_dict = state_dict['model']
        
    clean_state_dict = {}
    for k, v in state_dict.items():
        new_key = k
        while True:
            if new_key.startswith('_orig_mod.'):
                new_key = new_key[len('_orig_mod.'):]
            elif new_key.startswith('model.'):
                new_key = new_key[len('model.'):]
            else:
                break
        clean_state_dict[new_key] = v
            
    # Robust resizing of the puzzle embedding weights to preserve learned parameters
    puzzle_emb_name = "inner.puzzle_emb.weights"
    expected_shape = model.inner.puzzle_emb.weights.shape
    if puzzle_emb_name in clean_state_dict:
        puzzle_emb = clean_state_dict[puzzle_emb_name]
        if puzzle_emb.shape != expected_shape:
            print(f"  Resizing puzzle embedding. Found {puzzle_emb.shape}, Expected {expected_shape}")
            with torch.no_grad():
                new_weights = torch.empty(expected_shape, dtype=puzzle_emb.dtype, device=puzzle_emb.device)
                mean_emb = torch.mean(puzzle_emb.detach(), dim=0)
                new_weights[:] = mean_emb
                min_rows = min(puzzle_emb.shape[0], expected_shape[0])
                new_weights[:min_rows] = puzzle_emb[:min_rows].detach()
                clean_state_dict[puzzle_emb_name] = new_weights
            
    model.load_state_dict(clean_state_dict, strict=False)
    model.__dict__['model'] = model
    model.eval()
    print(f"✓ {name} successfully loaded!")
    return model

In [21]:
# Load checkpoints
arc_ckpt = "eval_checkpoint/step_10620"
maze_ckpt = "trm_maze_hard/model.pt"
sudoku_ckpt = "trm_sudoku_extreme/step_39060_sudoku_epoch_60k"

model_arc = load_trm_model("ARC 2024", arc_ckpt, arc_config)
model_maze = load_trm_model("Maze Hard", maze_ckpt, maze_config)
model_sudoku = load_trm_model("Sudoku Extreme", sudoku_ckpt, sudoku_config)

Loading ARC 2024 model from eval_checkpoint/step_10620...
✓ ARC 2024 successfully loaded!
Loading Maze Hard model from trm_maze_hard/model.pt...
✓ Maze Hard successfully loaded!
Loading Sudoku Extreme model from trm_sudoku_extreme/step_39060_sudoku_epoch_60k...
✓ Sudoku Extreme successfully loaded!


## Section 3 — Dataloaders Setup

In [22]:
class ARCDataset(Dataset):
    def __init__(self, split_dir: str):
        if not os.path.exists(split_dir):
            raise FileNotFoundError(f"Directory {split_dir} does not exist.")
        self.inputs = np.load(f"{split_dir}/all__inputs.npy")
        self.labels = np.load(f"{split_dir}/all__labels.npy")

        puzzle_ids  = np.load(f"{split_dir}/all__puzzle_identifiers.npy")
        puzzle_ptr  = np.load(f"{split_dir}/all__puzzle_indices.npy")

        counts = np.diff(puzzle_ptr).astype(np.int64)
        self.per_sample_pids = np.repeat(puzzle_ids, counts)

        assert len(self.inputs) == len(self.per_sample_pids), (
            f"Shape mismatch: inputs={len(self.inputs)}, pids={len(self.per_sample_pids)}"
        )
        meta_path = f"{split_dir}/../train/dataset.json"
        if os.path.exists(meta_path):
            with open(meta_path) as fj:
                meta = json.load(fj)
            self.seq_len = meta["seq_len"]
            self.vocab_size = meta["vocab_size"]
            self.num_puzzle_identifiers = meta["num_puzzle_identifiers"]
        else:
            self.seq_len = self.inputs.shape[1]

    def __len__(self): return len(self.inputs)

    def __getitem__(self, i):
        return (
            torch.tensor(self.inputs[i],          dtype=torch.long),
            torch.tensor(self.labels[i],           dtype=torch.long),
            torch.tensor(self.per_sample_pids[i],  dtype=torch.long),
        )

def build_loader(dir_path, batch_size=64):
    if not os.path.exists(dir_path):
        print(f"[WARNING] Path {dir_path} not found. Loader cannot be built.")
        return None
    ds = ARCDataset(dir_path)
    loader = DataLoader(ds, batch_size=batch_size, shuffle=False, num_workers=0)
    return loader

loader_arc = build_loader("./data/arc2test-aug-128/test")
loader_maze = build_loader("./data/maze-30x30-hard-1k/test")
loader_sudoku = build_loader("./data/sudoku-extreme-full/test", batch_size=512)

## Section 4 — Model-Specific Evaluators & Seed-loop Wrapper

In [23]:
from dataset.build_arc_dataset import inverse_aug, grid_hash, arc_grid_to_np, PuzzleIdSeparator
from evaluators.arc import _crop

DATA_DIR = "./data/arc2test-aug-128"

crop_cache = {}
def get_crop(seq):
    seq_bytes = seq.tobytes()
    if seq_bytes not in crop_cache:
        crop_cache[seq_bytes] = _crop(seq)
    return crop_cache[seq_bytes]

@torch.no_grad()
def evaluate_arc_trm(mdl, loader, device, n_sup_max=16, max_batches=None, return_pass2=False, fast_mode=True, trunc_len=None):
    puzzles_json_path = os.path.join(DATA_DIR, "test_puzzles.json")
    identifiers_json_path = os.path.join(DATA_DIR, "identifiers.json")
    if os.path.exists(puzzles_json_path):
        with open(puzzles_json_path) as f:
            test_puzzles = json.load(f)
    else:
        test_puzzles = {}
    if os.path.exists(identifiers_json_path):
        with open(identifiers_json_path) as f:
            identifier_map = json.load(f)
    else:
        identifier_map = []
        
    aug_cache = {}
    def get_aug(pid):
        if pid not in aug_cache:
            idx = int(pid)
            if 0 <= idx < len(identifier_map):
                name = identifier_map[idx]
                aug_cache[pid] = inverse_aug(name)
            else:
                aug_cache[pid] = (str(pid), lambda x: x)
        return aug_cache[pid]
        
    inner = get_inner(mdl)
    inner.eval()
    has_bnb = any(m.__class__.__name__ == 'Linear8bitLt' for m in inner.modules())
    if not has_bnb:
        inner = inner.to(device)
    
    ds = loader.dataset

    local_hmap = {}
    local_preds = {}
    inputs_np = ds.inputs
    labels_np = ds.labels
    pids_np = ds.per_sample_pids
    
    # In fast_mode, select exactly the canonical sample(s) for each unique original puzzle to be fast
    selected_sample_indices = None
    if fast_mode:
        selected_sample_indices = []
        for s_idx in range(len(pids_np)):
            pid = int(pids_np[s_idx])
            if pid == 0: continue
            name = identifier_map[pid]
            if "|||" not in name:
                selected_sample_indices.append(s_idx)
        if len(selected_sample_indices) == 0:
            seen_orig_names = {}
            for s_idx in range(len(pids_np)):
                pid = int(pids_np[s_idx])
                if pid == 0: continue
                name = identifier_map[pid]
                orig_name = name.split("|||")[0]
                if orig_name not in seen_orig_names:
                    seen_orig_names[orig_name] = pid
                if pids_np[s_idx] == seen_orig_names[orig_name]:
                    selected_sample_indices.append(s_idx)
        selected_sample_indices = np.array(selected_sample_indices, dtype=np.int32)
        inputs_np = inputs_np[selected_sample_indices]
        labels_np = labels_np[selected_sample_indices]
        pids_np = pids_np[selected_sample_indices]
        
    num_samples = len(inputs_np)
    batch_size = loader.batch_size if (hasattr(loader, "batch_size") and loader.batch_size is not None) else 512
    num_batches = math.ceil(num_samples / batch_size)
    t0 = time.time()
    
    for batch_idx in range(num_batches):
        if max_batches is not None and batch_idx >= max_batches:
            break
        start_idx = batch_idx * batch_size
        end_idx = min(start_idx + batch_size, num_samples)
        
        x_batch = torch.from_numpy(inputs_np[start_idx:end_idx]).to(device, dtype=torch.long)
        if trunc_len is not None and trunc_len < x_batch.shape[1]:
            x_batch = x_batch.clone()
            x_batch[:, trunc_len:] = 0
        y_true = torch.from_numpy(labels_np[start_idx:end_idx]).to(device, dtype=torch.long)
        pids = torch.from_numpy(pids_np[start_idx:end_idx]).to(device, dtype=torch.long)
        
        batch = {"inputs": x_batch.to(torch.int32), "labels": y_true.to(torch.int32), "puzzle_identifiers": pids.to(torch.int32)}
        carry = inner.initial_carry(batch)
        ic = carry.inner_carry
        cast = lambda t: t.to(device)
        carry = TinyRecursiveReasoningModel_ACTV1Carry(
            inner_carry=TinyRecursiveReasoningModel_ACTV1InnerCarry(z_H=cast(ic.z_H), z_L=cast(ic.z_L)),
            steps=carry.steps.to(device),
            halted=carry.halted.to(device),
            current_data={k: v.to(device) for k, v in carry.current_data.items()},
        )
        
        last_outputs = None
        for _ in range(n_sup_max):
            carry, outputs = inner(carry, batch)
            last_outputs = outputs
            if carry.halted.all():
                break
        if last_outputs is None: continue
        
        preds_batch = last_outputs["logits"].argmax(-1).cpu().numpy()
        q_logits = last_outputs.get("q_halt_logits", torch.zeros(preds_batch.shape[0], device=device))
        q_values = q_logits.sigmoid().cpu().numpy().flatten()
        
        inputs_cpu = inputs_np[start_idx:end_idx]
        pids_cpu = pids_np[start_idx:end_idx]
        
        for i in range(preds_batch.shape[0]):
            value = pids_cpu[i]
            if value == 0: continue
            orig_name, _inverse_fn = get_aug(value)
            pred_seq = preds_batch[i]
            q_val = float(q_values[i])
            
            inp_seq = inputs_cpu[i]
            input_grid = _inverse_fn(get_crop(inp_seq))
            input_hash = grid_hash(input_grid)
            pred_grid = _inverse_fn(get_crop(pred_seq))
            pred_hash = grid_hash(pred_grid)
            local_hmap[pred_hash] = pred_grid
            local_preds.setdefault(orig_name, {})
            local_preds[orig_name].setdefault(input_hash, [])
            local_preds[orig_name][input_hash].append((pred_hash, q_val))
            
    n_puzzles = 0
    correct = [0, 0]
    cell_hits, n_cells = 0, 0
    evaluated_puzzles = [name for name in test_puzzles.keys() if name in local_preds]
    for name in evaluated_puzzles:
        puzzle = test_puzzles[name]
        n_puzzles += 1
        num_correct = [0, 0]
        for pair in puzzle["test"]:
            inp_grid = arc_grid_to_np(pair["input"])
            out_grid = arc_grid_to_np(pair["output"])
            input_hash = grid_hash(inp_grid)
            label_hash = grid_hash(out_grid)
            p_map = {}
            for h, q in local_preds[name].get(input_hash, []):
                p_map.setdefault(h, [0, 0.0])
                p_map[h][0] += 1
                p_map[h][1] += q
            if not len(p_map): continue
            for h, stats in p_map.items():
                stats[1] /= stats[0]
            p_map_sorted = sorted(p_map.items(), key=lambda kv: (kv[1][0], kv[1][1]), reverse=True)
            if p_map_sorted[0][0] == label_hash: num_correct[0] = 1
            if any(h == label_hash for h, _ in p_map_sorted[:2]): num_correct[1] = 1
        correct[0] += num_correct[0]
        correct[1] += num_correct[1]
        
        for pair in puzzle["test"]:
            inp_grid = arc_grid_to_np(pair["input"])
            out_grid = arc_grid_to_np(pair["output"])
            input_hash = grid_hash(inp_grid)
            preds_list = local_preds.get(name, {}).get(input_hash, [])
            if not preds_list: continue
            p_map = {}
            for h, q in preds_list:
                p_map.setdefault(h, [0, 0.0])
                p_map[h][0] += 1
                p_map[h][1] += q
            for h, stats in p_map.items():
                stats[1] /= stats[0]
            p_map_sorted = sorted(p_map.items(), key=lambda kv: kv[1], reverse=True)
            top_hash = p_map_sorted[0][0]
            top_grid = local_hmap[top_hash]
            if top_grid.shape == out_grid.shape: cell_hits += (top_grid == out_grid).sum()
            n_cells += out_grid.size
            
    cell_acc = cell_hits / n_cells if n_cells > 0 else 0.0
    pass_1_acc = correct[0] / n_puzzles if n_puzzles > 0 else 0.0
    pass_2_acc = correct[1] / n_puzzles if n_puzzles > 0 else 0.0
    elapsed = time.time() - t0
    ms_per_puzzle = (elapsed / n_puzzles * 1000) if n_puzzles > 0 else 0.0
    if return_pass2:
        return pass_1_acc, pass_2_acc, cell_acc, ms_per_puzzle, n_puzzles
    else:
        return pass_1_acc, cell_acc, ms_per_puzzle, n_puzzles

@torch.no_grad()
def evaluate_batch_trm(mdl, loader, device, n_sup_max=16, max_batches=None, return_pass2=False, trunc_len=None):
    inner = get_inner(mdl)
    inner.eval()
    has_bnb = any(m.__class__.__name__ == 'Linear8bitLt' for m in inner.modules())
    if not has_bnb:
        inner = inner.to(device)
    total_samples, total_correct_cells, total_cells, total_exact_correct = 0, 0, 0, 0
    t0 = time.time()
    batch_idx = 0
    for inputs, labels, pids in loader:
        if max_batches is not None and batch_idx >= max_batches: break
        inputs, labels, pids = inputs.to(device), labels.to(device), pids.to(device)
        if trunc_len is not None and trunc_len < inputs.shape[1]:
            inputs = inputs.clone()
            inputs[:, trunc_len:] = 0
        batch = {"inputs": inputs.to(torch.int32), "labels": labels.to(torch.int32), "puzzle_identifiers": pids.to(torch.int32)}
        carry = inner.initial_carry(batch)
        ic = carry.inner_carry
        cast = lambda t: t.to(device)
        carry = TinyRecursiveReasoningModel_ACTV1Carry(
            inner_carry=TinyRecursiveReasoningModel_ACTV1InnerCarry(z_H=cast(ic.z_H), z_L=cast(ic.z_L)),
            steps=carry.steps.to(device),
            halted=carry.halted.to(device),
            current_data={k: v.to(device) for k, v in carry.current_data.items()},
        )
        for _ in range(n_sup_max):
            carry, outputs = inner(carry, batch)
            if carry.halted.all(): break
        preds = torch.argmax(outputs["logits"], dim=-1)
        mask = (labels != 0)
        is_correct = mask & (preds == labels)
        total_correct_cells += is_correct.sum().item()
        total_cells += mask.sum().item()
        loss_counts = mask.sum(-1)
        seq_is_correct = (is_correct.sum(-1) == loss_counts) & (loss_counts > 0)
        total_exact_correct += seq_is_correct.sum().item()
        total_samples += inputs.shape[0]
        batch_idx += 1
        
    elapsed = time.time() - t0
    cell_acc = total_correct_cells / total_cells if total_cells > 0 else 0.0
    pass_1_acc = total_exact_correct / total_samples if total_samples > 0 else 0.0
    ms_per_puzzle = (elapsed / total_samples * 1000) if total_samples > 0 else 0.0
    if return_pass2:
        return pass_1_acc, pass_1_acc, cell_acc, ms_per_puzzle, total_samples
    else:
        return pass_1_acc, cell_acc, ms_per_puzzle, total_samples

def evaluate_across_seeds(model_name, mdl, loader, seeds=[42], **kwargs):
    results = []
    if loader is None:
        return [{ "seed": s, "exact1": 0.0, "exact2": 0.0, "cell": 0.0, "ms": 0.0 } for s in seeds]
    for seed in seeds:
        torch.manual_seed(seed)
        np.random.seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
        if "arc" in model_name.lower():
            metrics = evaluate_arc_trm(mdl, loader, device=str(DEVICE), return_pass2=True, **kwargs)
        else:
            metrics = evaluate_batch_trm(mdl, loader, device=str(DEVICE), return_pass2=True, **kwargs)
        r = {
            "seed": seed,
            "exact1": metrics[0],
            "exact2": metrics[1],
            "cell": metrics[2],
            "ms": metrics[3]
        }
        results.append(r)
        print(f"  Seed {seed}: Pass@1 = {r['exact1']:.4f}, Cell Acc = {r['cell']:.4f}, Latency = {r['ms']:.2f} ms")
    return results

In [24]:
# # ── 4.1  Sudoku Extreme DataLoader from pre-built .npy files ────────────────────────
# import os
# import csv
# import json
# import math
# import numpy as np
# import torch
# from torch.utils.data import Dataset, DataLoader

# DATA_DIR = "./data/sudoku-extreme-full"

# # Automatically download and build dataset if missing
# if not os.path.exists(os.path.join(DATA_DIR, "test", "all__inputs.npy")):
#     print("Dataset not found. Downloading and building Sudoku Extreme dataset from HuggingFace...")
#     try:
#         from huggingface_hub import hf_hub_download
        
#         try:
#             from tqdm import tqdm
#         except Exception:
#             def tqdm(iterable, *args, **kwargs):
#                 return iterable
        
#         source_repo = "sapientinc/sudoku-extreme"
        
#         def convert_subset(split_name: str, max_samples=1000):
#             print(f"Downloading and processing '{split_name}' split...")
#             inputs = []
#             labels = []
            
#             # Download from HuggingFace Hub
#             csv_path = hf_hub_download(source_repo, f"{split_name}.csv", repo_type="dataset")
            
#             with open(csv_path, newline="") as csvfile:
#                 reader = csv.reader(csvfile)
#                 next(reader)  # Skip header
#                 count = 0
#                 for row in reader:
#                     # columns: source, q, a, rating
#                     if len(row) < 3:
#                         continue
#                     q, a = row[1], row[2]
#                     if max_samples is not None and count >= max_samples:
#                         break
#                     assert len(q) == 81 and len(a) == 81
#                     q_clean = q.replace('.', '0')
#                     inputs.append(np.frombuffer(q_clean.encode(), dtype=np.uint8).reshape(9, 9) - ord('0'))
#                     labels.append(np.frombuffer(a.encode(), dtype=np.uint8).reshape(9, 9) - ord('0'))
#                     count += 1
                    
#             results = {k: [] for k in ["inputs", "labels", "puzzle_identifiers", "puzzle_indices", "group_indices"]}
#             puzzle_id = 0
#             example_id = 0
#             results["puzzle_indices"].append(0)
#             results["group_indices"].append(0)
            
#             pbar = tqdm(zip(inputs, labels), total=len(inputs), desc=f"Converting {split_name}")
#             for inp, out in pbar:
#                 results["inputs"].append(inp)
#                 results["labels"].append(out)
#                 example_id += 1
#                 puzzle_id += 1
#                 results["puzzle_indices"].append(example_id)
#                 results["puzzle_identifiers"].append(0)
#                 results["group_indices"].append(puzzle_id)
                
#             def _seq_to_numpy(seq):
#                 arr = np.concatenate(seq).reshape(len(seq), -1)
#                 assert np.all((arr >= 0) & (arr <= 9))
#                 return arr + 1
                
#             final_results = {
#                 "inputs": _seq_to_numpy(results["inputs"]),
#                 "labels": _seq_to_numpy(results["labels"]),
#                 "group_indices": np.array(results["group_indices"], dtype=np.int32),
#                 "puzzle_indices": np.array(results["puzzle_indices"], dtype=np.int32),
#                 "puzzle_identifiers": np.array(results["puzzle_identifiers"], dtype=np.int32),
#             }
            
#             metadata = {
#                 "seq_len": 81,
#                 "vocab_size": 11,
#                 "pad_id": 0,
#                 "ignore_label_id": 0,
#                 "blank_identifier_id": 0,
#                 "num_puzzle_identifiers": 1,
#                 "total_groups": len(final_results["group_indices"]) - 1,
#                 "mean_puzzle_examples": 1,
#                 "total_puzzles": len(final_results["group_indices"]) - 1,
#                 "sets": ["all"]
#             }
            
#             save_dir = os.path.join(DATA_DIR, split_name)
#             os.makedirs(save_dir, exist_ok=True)
            
#             with open(os.path.join(save_dir, "dataset.json"), "w") as f:
#                 json.dump(metadata, f)
                
#             for k, v in final_results.items():
#                 np.save(os.path.join(save_dir, f"all__{k}.npy"), v)
                
#         # Generate train and test splits (limit to 1000 for fast eval)
#         os.makedirs(DATA_DIR, exist_ok=True)
#         convert_subset("train", max_samples=1000)
#         convert_subset("test", max_samples=1000)
        
#         with open(os.path.join(DATA_DIR, "identifiers.json"), "w") as f:
#             json.dump(["<blank>"], f)
            
#         print("✓ Sudoku Extreme dataset successfully downloaded and generated in:", DATA_DIR)
#     except Exception as e:
#         print(f"Error generating dataset: {e}")

# print(f"Using dataset directory: {DATA_DIR}")

# class ARCDataset(Dataset):
#     """
#     Loads the pre-built Sudoku dataset from numpy arrays.
#     """
#     def __init__(self, split_dir: str):
#         self.inputs = np.load(f"{split_dir}/all__inputs.npy")
#         self.labels = np.load(f"{split_dir}/all__labels.npy")

#         puzzle_ids  = np.load(f"{split_dir}/all__puzzle_identifiers.npy")  # (N_puzzles,)
#         puzzle_ptr  = np.load(f"{split_dir}/all__puzzle_indices.npy")       # (N_puzzles+1,)

#         # CSR expansion: each sample gets its puzzle's identifier
#         counts = np.diff(puzzle_ptr).astype(np.int64)           # samples per puzzle
#         self.per_sample_pids = np.repeat(puzzle_ids, counts)    # (N_samples,)

#         assert len(self.inputs) == len(self.per_sample_pids), (
#             f"Shape mismatch: inputs={len(self.inputs)}, pids={len(self.per_sample_pids)}"
#         )

#         with open(f"{split_dir}/../train/dataset.json") as fj:
#             meta = json.load(fj)
#         self.seq_len               = meta["seq_len"]
#         self.vocab_size            = meta["vocab_size"]
#         self.num_puzzle_identifiers = meta["num_puzzle_identifiers"]

#     def __len__(self): return len(self.inputs)

#     def __getitem__(self, i):
#         return (
#             torch.tensor(self.inputs[i],          dtype=torch.long),
#             torch.tensor(self.labels[i],           dtype=torch.long),
#             torch.tensor(self.per_sample_pids[i],  dtype=torch.long),
#         )

# SudokuDataset = ARCDataset

# try:
#     test_ds  = ARCDataset(f"{DATA_DIR}/test")
#     train_ds = ARCDataset(f"{DATA_DIR}/train")
#     BATCH_SIZE = 512
#     test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
#     train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
#     print(f"Test  split : {len(test_ds):,} samples  (seq_len={test_ds.seq_len})")
#     print(f"Train split : {len(train_ds):,} samples")
    
#     # Simple check on one batch to verify
#     for x, y, pids in test_loader:
#         print("Dataset loaded successfully!")
#         break
# except Exception as e:
#     print(f"Error loading dataset: {e}")


## Section 5 — Quantization Wrappers & Stability Sweeps

In [25]:
from models.layers import CastedLinear
def quantize_fp16(mdl):
    return copy.deepcopy(get_inner(mdl)).cuda().half()

def quantize_int8_bnb(mdl):
    try:
        import bitsandbytes as bnb
        from bitsandbytes.nn import Linear8bitLt
    except ImportError:
        print("bitsandbytes not installed. Skipping INT8 bnb.")
        return mdl
    m = copy.deepcopy(get_inner(mdl)).cuda()
    for name, module in list(m.named_modules()):
        if not isinstance(module, (nn.Linear, CastedLinear)): continue
        if "lm_head" in name or "q_head" in name or "mlp_t" in name: continue
        out_f, in_f = module.weight.shape
        new_layer = Linear8bitLt(in_f, out_f, bias=module.bias is not None, has_fp16_weights=False, threshold=6.0).cuda()
        new_layer.weight = bnb.nn.Int8Params(module.weight.data.to(torch.float16), requires_grad=False, has_fp16_weights=False)
        if module.bias is not None:
            new_layer.bias = nn.Parameter(module.bias.data.to(torch.float16))
        parts = name.split(".")
        parent = m
        for p in parts[:-1]: parent = getattr(parent, p)
        setattr(parent, parts[-1], new_layer)
    return m

class FakeQuantINT4(nn.Module):
    def __init__(self, weight, bias=None):
        super().__init__()
        self.weight = nn.Parameter(weight.clone(), requires_grad=False)
        self.bias = nn.Parameter(bias.clone(), requires_grad=False) if bias is not None else None
    def _fake_quant(self, x):
        q_max = 7
        scale = x.float().abs().max().clamp(min=1e-8) / q_max
        x_q = torch.clamp((x.float() / scale).round(), -q_max, q_max)
        return (x_q * scale).to(x.dtype)
    def forward(self, x):
        w_q = self._fake_quant(self.weight).to(x.dtype)
        b = self.bias.to(x.dtype) if self.bias is not None else None
        return F.linear(x, w_q, b)

def quantize_int4_fake(mdl):
    m = copy.deepcopy(get_inner(mdl))
    for name, module in list(m.named_modules()):
        if not isinstance(module, (nn.Linear, CastedLinear)): continue
        parts = name.split(".")
        parent = m
        for p in parts[:-1]: parent = getattr(parent, p)
        setattr(parent, parts[-1], FakeQuantINT4(module.weight, module.bias))
    return m

models_list = [
    ("ARC 2024", model_arc, loader_arc, 10),
    ("Maze Hard", model_maze, loader_maze, 10),
    ("Sudoku Extreme", model_sudoku, loader_sudoku, 10)
]

quant_methods = [
    ("FP32", lambda m: m),
    ("FP16", quantize_fp16),
    ("INT8 (bnb)", quantize_int8_bnb),
    ("INT4 (fake)", quantize_int4_fake)
]

records = []
for model_name, model_obj, loader, n_sup in models_list:
    for q_name, q_fn in quant_methods:
        print(f"Evaluating {model_name} with {q_name} quantization...")
        q_model = q_fn(model_obj)
        res = evaluate_across_seeds(model_name, q_model, loader, n_sup_max=n_sup)
        for r in res:
            records.append({
                "Model": model_name, "Quant": q_name, "Seed": r["seed"], "Pass@1": r["exact1"], "Cell Acc": r["cell"], "Latency (ms)": r["ms"]
            })
df_quant = pd.DataFrame(records)
df_quant.pivot(index=["Model", "Quant"], columns="Seed", values=["Pass@1", "Cell Acc"])

Evaluating ARC 2024 with FP32 quantization...
  Seed 42: Pass@1 = 0.3600, Cell Acc = 0.8822, Latency = 20.28 ms
Evaluating ARC 2024 with FP16 quantization...
  Seed 42: Pass@1 = 0.3600, Cell Acc = 0.8757, Latency = 70.93 ms
Evaluating ARC 2024 with INT8 (bnb) quantization...
  Seed 42: Pass@1 = 0.3600, Cell Acc = 0.8762, Latency = 55.31 ms
Evaluating ARC 2024 with INT4 (fake) quantization...
  Seed 42: Pass@1 = 0.2550, Cell Acc = 0.8480, Latency = 20.86 ms
Evaluating Maze Hard with FP32 quantization...
  Seed 42: Pass@1 = 0.8680, Cell Acc = 0.9952, Latency = 19.17 ms
Evaluating Maze Hard with FP16 quantization...
  Seed 42: Pass@1 = 0.8700, Cell Acc = 0.9953, Latency = 68.01 ms
Evaluating Maze Hard with INT8 (bnb) quantization...
  Seed 42: Pass@1 = 0.8690, Cell Acc = 0.9953, Latency = 46.99 ms
Evaluating Maze Hard with INT4 (fake) quantization...
  Seed 42: Pass@1 = 0.8640, Cell Acc = 0.9950, Latency = 19.79 ms
Evaluating Sudoku Extreme with FP32 quantization...
  Seed 42: Pass@1 = 0.

Pass@1  Cell Acc
Seed                           42        42
Model          Quant                       
ARC 2024       FP16         0.360  0.875664
               FP32         0.360  0.882241
               INT4 (fake)  0.255  0.848013
               INT8 (bnb)   0.360  0.876181
Maze Hard      FP16         0.870  0.995290
               FP32         0.868  0.995237
               INT4 (fake)  0.864  0.995008
               INT8 (bnb)   0.869  0.995272
Sudoku Extreme FP16         0.685  0.871975
               FP32         0.691  0.874716
               INT4 (fake)  0.053  0.660235
               INT8 (bnb)   0.691  0.875074

## Section 6 — Recursive Depth × Quantization Grid

In [26]:
depth_sweeps = [
    (1, 1), (1, 2), (1, 4), (1, 8), (1, 10),
    (2, 4), (2, 8), (2, 10),
    (3, 4), (3, 8), (3, 10),
    (4, 4), (4, 8), (4, 10)
]
sweep_records = []
for model_name, model_obj, loader, _ in models_list:
    for H, n_sup in depth_sweeps:
        print(f"Sweeping {model_name} with H={H}, n_sup={n_sup}...")
        inner = get_inner(model_obj)
        orig_H = inner.config.H_cycles
        inner.config.H_cycles = H
        res = evaluate_across_seeds(model_name, model_obj, loader, n_sup_max=n_sup)
        inner.config.H_cycles = orig_H
        for r in res:
            sweep_records.append({
                "Model": model_name, "H": H, "n_sup": n_sup, "Seed": r["seed"], "Pass@1": r["exact1"], "Cell Acc": r["cell"]
            })
df_sweep = pd.DataFrame(sweep_records)
df_sweep.pivot(index=["Model", "H", "n_sup"], columns="Seed", values="Pass@1")

Sweeping ARC 2024 with H=1, n_sup=1...
  Seed 42: Pass@1 = 0.0025, Cell Acc = 0.5452, Latency = 1.09 ms
Sweeping ARC 2024 with H=1, n_sup=2...
  Seed 42: Pass@1 = 0.0750, Cell Acc = 0.7953, Latency = 1.78 ms
Sweeping ARC 2024 with H=1, n_sup=4...
  Seed 42: Pass@1 = 0.3450, Cell Acc = 0.8824, Latency = 3.08 ms
Sweeping ARC 2024 with H=1, n_sup=8...
  Seed 42: Pass@1 = 0.3550, Cell Acc = 0.8831, Latency = 5.82 ms
Sweeping ARC 2024 with H=1, n_sup=10...
  Seed 42: Pass@1 = 0.3575, Cell Acc = 0.8761, Latency = 7.08 ms
Sweeping ARC 2024 with H=2, n_sup=4...
  Seed 42: Pass@1 = 0.3550, Cell Acc = 0.8831, Latency = 5.74 ms
Sweeping ARC 2024 with H=2, n_sup=8...
  Seed 42: Pass@1 = 0.3600, Cell Acc = 0.8767, Latency = 10.98 ms
Sweeping ARC 2024 with H=2, n_sup=10...
  Seed 42: Pass@1 = 0.3600, Cell Acc = 0.8755, Latency = 13.70 ms
Sweeping ARC 2024 with H=3, n_sup=4...
  Seed 42: Pass@1 = 0.3575, Cell Acc = 0.8767, Latency = 8.32 ms
Sweeping ARC 2024 with H=3, n_sup=8...
  Seed 42: Pass@1 = 0

Seed                        42
Model          H n_sup        
ARC 2024       1 1      0.0025
                 2      0.0750
                 4      0.3450
                 8      0.3550
                 10     0.3575
               2 4      0.3550
                 8      0.3600
                 10     0.3600
               3 4      0.3575
                 8      0.3600
                 10     0.3600
               4 4      0.3600
                 8      0.3625
                 10     0.3600
Maze Hard      1 1      0.0000
                 2      0.1390
                 4      0.8450
                 8      0.8680
                 10     0.8680
               2 4      0.8680
                 8      0.8680
                 10     0.8680
               3 4      0.8680
                 8      0.8680
                 10     0.8680
               4 4      0.8680
                 8      0.8680
                 10     0.8680
Sudoku Extreme 1 1      0.0060
                 2      0.0210
                 4      0.3810
                 8      0.5410
                 10     0.5760
               2 4      0.5410
                 8      0.6370
                 10     0.6630
               3 4      0.6060
                 8      0.6790
                 10     0.6910
               4 4      0.6370
                 8      0.6930
                 10     0.7030

## Section 7 — Recursive State Similarity Analysis

In [31]:
def hook_carry_similarity(model_obj, loader, device):
    inner = get_inner(model_obj)
    if hasattr(inner, '_forward_hooks'):
        inner._forward_hooks.clear()
    if hasattr(inner, 'inner') and hasattr(inner.inner, 'L_level') and hasattr(inner.inner.L_level, '_forward_hooks'):
        inner.inner.L_level._forward_hooks.clear()
    similarities = []
    def hook_fn(module, inputs, outputs):
        next_carry = outputs[0]
        z_H = next_carry.inner_carry.z_H
        if hasattr(hook_fn, "prev_z_H") and hook_fn.prev_z_H is not None:
            cos_sim = F.cosine_similarity(z_H.flatten(), hook_fn.prev_z_H.flatten(), dim=0)
            similarities.append(cos_sim.item())
        hook_fn.prev_z_H = z_H.clone()
    hook_fn.prev_z_H = None
    handle = inner.register_forward_hook(hook_fn)
    batch = next(iter(loader))
    inputs, labels, pids = [t.to(device) for t in batch]
    batch_dict = {"inputs": inputs.to(torch.int32), "labels": labels.to(torch.int32), "puzzle_identifiers": pids.to(torch.int32)}
    carry = inner.initial_carry(batch_dict)
    from models.recursive_reasoning.trm import TinyRecursiveReasoningModel_ACTV1Carry, TinyRecursiveReasoningModel_ACTV1InnerCarry
    ic = carry.inner_carry
    carry = TinyRecursiveReasoningModel_ACTV1Carry(
        inner_carry=TinyRecursiveReasoningModel_ACTV1InnerCarry(z_H=ic.z_H.to(device), z_L=ic.z_L.to(device)),
        steps=carry.steps.to(device),
        halted=carry.halted.to(device),
        current_data={k: v.to(device) for k, v in carry.current_data.items()},
    )
    for _ in range(8):
        carry, _ = inner(carry, batch_dict)
    handle.remove()
    return np.mean(similarities) if similarities else 1.0

sim_records = []
for model_name, model_obj, loader, _ in models_list:
    if loader is None: continue
    for q_name, q_fn in quant_methods:
        print(f"Calculating carry similarity for {model_name} with {q_name}...")
        q_model = q_fn(model_obj)
        seeds_sim = []
        for seed in [42, 43, 44]:
            torch.manual_seed(seed)
            np.random.seed(seed)
            seeds_sim.append(hook_carry_similarity(q_model, loader, DEVICE))
        mean_sim = np.mean(seeds_sim)
        std_sim = np.std(seeds_sim)
        print(f"  Similarity: {mean_sim:.4f} ± {std_sim:.4f}")
        sim_records.append({
            "Model": model_name, "Quant": q_name, "Mean Similarity": mean_sim, "Std Similarity": std_sim
        })
df_sim = pd.DataFrame(sim_records)
df_sim

Calculating carry similarity for ARC 2024 with FP32...
  Similarity: 0.9827 ± 0.0000
Calculating carry similarity for ARC 2024 with FP16...
  Similarity: 0.9804 ± 0.0000
Calculating carry similarity for ARC 2024 with INT8 (bnb)...
  Similarity: 0.9816 ± 0.0000
Calculating carry similarity for ARC 2024 with INT4 (fake)...
  Similarity: 0.9860 ± 0.0000
Calculating carry similarity for Maze Hard with FP32...
  Similarity: 0.9950 ± 0.0000
Calculating carry similarity for Maze Hard with FP16...
  Similarity: 0.9940 ± 0.0000
Calculating carry similarity for Maze Hard with INT8 (bnb)...
  Similarity: 0.9955 ± 0.0000
Calculating carry similarity for Maze Hard with INT4 (fake)...
  Similarity: 0.9955 ± 0.0000
Calculating carry similarity for Sudoku Extreme with FP32...
  Similarity: 0.9180 ± 0.0000
Calculating carry similarity for Sudoku Extreme with FP16...
  Similarity: 0.9235 ± 0.0000
Calculating carry similarity for Sudoku Extreme with INT8 (bnb)...
  Similarity: 0.9185 ± 0.0000
Calculating

,Model,Quant,Mean Similarity,Std Similarity
0,ARC 2024,FP32,0.982701,0.000000e+00
1,ARC 2024,FP16,0.980448,0.000000e+00
2,ARC 2024,INT8 (bnb),0.981585,1.110223e-16
3,ARC 2024,INT4 (fake),0.986049,0.000000e+00
4,Maze Hard,FP32,0.994978,1.110223e-16
5,Maze Hard,FP16,0.993982,0.000000e+00
6,Maze Hard,INT8 (bnb),0.995536,0.000000e+00
7,Maze Hard,INT4 (fake),0.995536,0.000000e+00
8,Sudoku Extreme,FP32,0.917969,0.000000e+00
9,Sudoku Extreme,FP16,0.923541,0.000000e+00


## Section 8 — Model Size & SRAM Footprint Estimator

In [32]:
def estimate_size_kb(mdl, bits):
    n = sum(p.numel() for p in get_inner(mdl).parameters())
    return n * bits / 8 / 1024

footprints = []
for model_name, model_obj, _, _ in models_list:
    inner = get_inner(model_obj)
    emb_w = inner.puzzle_emb.weights.shape
    num_emb_elements = emb_w[0] * emb_w[1]
    fp32_kb = estimate_size_kb(model_obj, 32)
    int8_kb = estimate_size_kb(model_obj, 8)
    int4_kb = estimate_size_kb(model_obj, 4)
    emb_fp32_kb = num_emb_elements * 32 / 8 / 1024
    single_puzzle_row_kb = emb_w[1] * 32 / 8 / 1024
    footprints.append({
        "Model": model_name, "Backbone FP32 (KB)": fp32_kb, "Backbone INT8 (KB)": int8_kb, "Backbone INT4 (KB)": int4_kb,
        "Full Embedding FP32 (KB)": emb_fp32_kb, "Single-puzzle Row (KB)": single_puzzle_row_kb
    })
df_foot = pd.DataFrame(footprints)
df_foot

,Model,Backbone FP32 (KB),Backbone INT8 (KB),Backbone INT4 (KB),Full Embedding FP32 (KB),Single-puzzle Row (KB)
0,ARC 2024,26676.007812,6669.001953,3334.500977,101822.0,2.0
1,Maze Hard,26652.007812,6663.001953,3331.500977,2.0,2.0
2,Sudoku Extreme,19644.007812,4911.001953,2455.500977,2.0,2.0


## Section 9 — Puzzle Embedding Compression

In [33]:
class INT8PuzzleEmbedding(nn.Module):
    def __init__(self, weight):
        super().__init__()
        self.weight_shape = weight.shape
        scale = weight.abs().max(dim=-1, keepdim=True).values.clamp(min=1e-8) / 127
        self.register_buffer("weight_q", torch.clamp((weight / scale).round(), -128, 127).to(torch.int8))
        self.register_buffer("scale", scale)
    def forward(self, idx):
        w = self.weight_q[idx].float() * self.scale[idx]
        return w

class SVDPuzzleEmbedding(nn.Module):
    def __init__(self, weight, rank=16):
        super().__init__()
        U, S, V = torch.linalg.svd(weight, full_matrices=False)
        self.U = nn.Parameter(U[:, :rank], requires_grad=False)
        self.S_V = nn.Parameter(torch.diag(S[:rank]) @ V[:rank, :], requires_grad=False)
    def forward(self, idx):
        return self.U[idx] @ self.S_V

class SinglePuzzleEmbedding(nn.Module):
    def __init__(self, row):
        super().__init__()
        self.row = nn.Parameter(row.clone(), requires_grad=False)
    def forward(self, idx):
        return self.row.expand(idx.shape[0], -1)

embedding_records = []
for model_name, model_obj, _, _ in models_list:
    inner = get_inner(model_obj)
    w = inner.puzzle_emb.weights.data
    int8_emb = INT8PuzzleEmbedding(w)
    w_int8 = int8_emb(torch.arange(w.shape[0], device=w.device))
    sim_int8 = F.cosine_similarity(w.flatten(), w_int8.flatten(), dim=0).item()
    
    svd_emb = SVDPuzzleEmbedding(w, rank=16)
    w_svd = svd_emb(torch.arange(w.shape[0], device=w.device))
    sim_svd = F.cosine_similarity(w.flatten(), w_svd.flatten(), dim=0).item()
    
    embedding_records.append({
        "Model": model_name,
        "INT8 Cos Sim": sim_int8,
        "SVD r=16 Cos Sim": sim_svd
    })
df_emb_comp = pd.DataFrame(embedding_records)
df_emb_comp

,Model,INT8 Cos Sim,SVD r=16 Cos Sim
0,ARC 2024,0.999974,0.729975
1,Maze Hard,0.999987,1.000000
2,Sudoku Extreme,0.999980,1.000000


## Section 10 — Fixed Evaluation: Per-Puzzle Aggregation

In [34]:
eval_records = []
for model_name, model_obj, loader, n_sup in models_list:
    if loader is None: continue
    print(f"Running baseline vs INT8-emb evaluation for {model_name}...")
    res_base = evaluate_across_seeds(model_name, model_obj, loader, n_sup_max=n_sup)
    
    m_int8 = copy.deepcopy(model_obj)
    inner = get_inner(m_int8)
    inner.puzzle_emb = INT8PuzzleEmbedding(inner.puzzle_emb.weights.data)
    res_int8_emb = evaluate_across_seeds(model_name, m_int8, loader, n_sup_max=n_sup)
    
    for r1, r2 in zip(res_base, res_int8_emb):
        print(f"  Seed {r1['seed']}: Baseline Pass@1 = {r1['exact1']:.4f}, INT8-emb Pass@1 = {r2['exact1']:.4f}")
        eval_records.append({
            "Model": model_name, "Seed": r1["seed"], "Baseline Pass@1": r1["exact1"], "INT8-emb Pass@1": r2["exact1"]
        })
df_agg = pd.DataFrame(eval_records)
df_agg

Running baseline vs INT8-emb evaluation for ARC 2024...
  Seed 42: Pass@1 = 0.3600, Cell Acc = 0.8822, Latency = 20.26 ms
  Seed 42: Pass@1 = 0.3600, Cell Acc = 0.8822, Latency = 20.25 ms
  Seed 42: Baseline Pass@1 = 0.3600, INT8-emb Pass@1 = 0.3600
Running baseline vs INT8-emb evaluation for Maze Hard...
  Seed 42: Pass@1 = 0.8680, Cell Acc = 0.9952, Latency = 19.24 ms
  Seed 42: Pass@1 = 0.8680, Cell Acc = 0.9952, Latency = 19.25 ms
  Seed 42: Baseline Pass@1 = 0.8680, INT8-emb Pass@1 = 0.8680
Running baseline vs INT8-emb evaluation for Sudoku Extreme...
  Seed 42: Pass@1 = 0.6910, Cell Acc = 0.8747, Latency = 3.08 ms
  Seed 42: Pass@1 = 0.6910, Cell Acc = 0.8747, Latency = 3.08 ms
  Seed 42: Baseline Pass@1 = 0.6910, INT8-emb Pass@1 = 0.6910


,Model,Seed,Baseline Pass@1,INT8-emb Pass@1
0,ARC 2024,42,0.360,0.360
1,Maze Hard,42,0.868,0.868
2,Sudoku Extreme,42,0.691,0.691


## Section 11 — Calibrated INT4 Quantization

In [35]:
class CalibratedFakeQuantINT4(nn.Module):
    def __init__(self, weight, bias=None):
        super().__init__()
        self.weight = nn.Parameter(weight.clone(), requires_grad=False)
        self.bias = nn.Parameter(bias.clone(), requires_grad=False) if bias is not None else None
        self.register_buffer("scale", torch.ones(weight.shape[0], 1))
        self.register_buffer("zero_point", torch.zeros(weight.shape[0], 1))
        self.calibrated = False
    def calibrate(self):
        q_max = 7
        w = self.weight.float()
        for i in range(w.shape[0]):
            w_i = w[i]
            w_min, w_max = w_i.min().item(), w_i.max().item()
            w_range = max(w_max - w_min, 1e-8)
            self.scale[i] = w_range / (2 * q_max)
            self.zero_point[i] = round(((w_max + w_min) / 2) / self.scale[i].item())
        self.calibrated = True
    def forward(self, x):
        if not self.calibrated:
            self.calibrate()
        w_q = torch.clamp(torch.round(self.weight.float() / self.scale) - self.zero_point, -7, 7)
        w_deq = (w_q + self.zero_point) * self.scale
        b = self.bias.to(x.dtype) if self.bias is not None else None
        return F.linear(x, w_deq.to(x.dtype), b)

def quantize_calibrated_int4(mdl):
    m = copy.deepcopy(get_inner(mdl))
    for name, module in list(m.named_modules()):
        if not isinstance(module, (nn.Linear, CastedLinear)): continue
        parts = name.split(".")
        parent = m
        for p in parts[:-1]: parent = getattr(parent, p)
        new_layer = CalibratedFakeQuantINT4(module.weight, module.bias)
        new_layer.calibrate()
        setattr(parent, parts[-1], new_layer)
    return m

cal_records = []
for model_name, model_obj, loader, n_sup in models_list:
    print(f"Evaluating Calibrated INT4 for {model_name}...")
    q_cal = quantize_calibrated_int4(model_obj)
    res = evaluate_across_seeds(model_name, q_cal, loader, n_sup_max=n_sup)
    for r in res:
        print(f"  Seed {r['seed']}: Pass@1 = {r['exact1']:.4f}, Cell Acc = {r['cell']:.4f}")
        cal_records.append({
            "Model": model_name, "Seed": r["seed"], "Calibrated INT4 Pass@1": r["exact1"], "Cell Acc": r["cell"]
        })
df_cal = pd.DataFrame(cal_records)
df_cal

Evaluating Calibrated INT4 for ARC 2024...
  Seed 42: Pass@1 = 0.3575, Cell Acc = 0.8919, Latency = 21.17 ms
  Seed 42: Pass@1 = 0.3575, Cell Acc = 0.8919
Evaluating Calibrated INT4 for Maze Hard...
  Seed 42: Pass@1 = 0.8710, Cell Acc = 0.9953, Latency = 19.80 ms
  Seed 42: Pass@1 = 0.8710, Cell Acc = 0.9953
Evaluating Calibrated INT4 for Sudoku Extreme...
  Seed 42: Pass@1 = 0.6750, Cell Acc = 0.8670, Latency = 3.15 ms
  Seed 42: Pass@1 = 0.6750, Cell Acc = 0.8670


,Model,Seed,Calibrated INT4 Pass@1,Cell Acc
0,ARC 2024,42,0.3575,0.891874
1,Maze Hard,42,0.8710,0.995339
2,Sudoku Extreme,42,0.6750,0.866951


## Section 12 — Quantization-Aware Fine-tuning (QAT)

In [36]:
def run_qat_loop(model_obj, loader, steps=5, micro_batch_size=64):
    """
    Performs a mini QAT training loop and returns validation loss/accuracy.
    """
    m = copy.deepcopy(get_inner(model_obj))
    optimizer = torch.optim.Adam(m.parameters(), lr=1e-5)
    m.train()
    losses = []
    
    batch = next(iter(loader))
    inputs, labels, pids = [t.to(DEVICE) for t in batch]
    num_micro_batches = max(1, len(inputs) // micro_batch_size)
    
    for _ in range(steps):
        optimizer.zero_grad()
        mb_loss = 0.0
        for mb_idx in range(num_micro_batches):
            mb_start = mb_idx * micro_batch_size
            mb_end = mb_start + micro_batch_size
            mb_x = inputs[mb_start:mb_end]
            mb_y = labels[mb_start:mb_end]
            mb_pids = pids[mb_start:mb_end]
            
            batch_dict = {"inputs": mb_x.to(torch.int32), "labels": mb_y.to(torch.int32), "puzzle_identifiers": mb_pids.to(torch.int32)}
            carry = m.initial_carry(batch_dict)
            carry, outputs = m(carry, batch_dict)
            loss = F.cross_entropy(outputs["logits"].flatten(0, 1), mb_y.flatten())
            mb_loss = mb_loss + loss
            
        mb_loss_normalized = mb_loss / num_micro_batches
        mb_loss_normalized.backward()
        optimizer.step()
        losses.append(mb_loss_normalized.item())
    return np.mean(losses)

# QAT Loop execution commented out by default
# qat_records = []
# for model_name, model_obj, loader, _ in models_list:
#     if loader is None: continue
#     for seed in [42, 43, 44]:
#         torch.manual_seed(seed)
#         np.random.seed(seed)
#         mean_loss = run_qat_loop(model_obj, loader, steps=5, micro_batch_size=64)
#         qat_records.append({
#             "Model": model_name, "Seed": seed, "QAT Loss": mean_loss
#         })
# df_qat = pd.DataFrame(qat_records)
# df_qat

## Section 13 — Structured Pruning

In [37]:
def prune_structured(mdl, amount=0.25):
    m = copy.deepcopy(get_inner(mdl))
    for name, param in m.named_parameters():
        if "weight" in name and len(param.shape) >= 2:
            norms = torch.norm(param, p=1, dim=1)
            threshold = torch.quantile(norms, amount)
            mask = norms >= threshold
            param.data[~mask] = 0.0
    return m

prune_records = []
for model_name, model_obj, loader, n_sup in models_list:
    for amount in [0.0, 0.25, 0.50]:
        print(f"Evaluating structured pruning for {model_name} (ratio={amount})...")
        pruned_mdl = prune_structured(model_obj, amount=amount)
        res = evaluate_across_seeds(model_name, pruned_mdl, loader, n_sup_max=n_sup)
        for r in res:
            print(f"  Seed {r['seed']}: Pass@1 = {r['exact1']:.4f}, Cell Acc = {r['cell']:.4f}")
            prune_records.append({
                "Model": model_name, "Prune Ratio": amount, "Seed": r["seed"], "Pass@1": r["exact1"], "Cell Acc": r["cell"]
            })
df_prune = pd.DataFrame(prune_records)
df_prune.pivot(index=["Model", "Prune Ratio"], columns="Seed", values="Pass@1")

Evaluating structured pruning for ARC 2024 (ratio=0.0)...
  Seed 42: Pass@1 = 0.3600, Cell Acc = 0.8822, Latency = 20.28 ms
  Seed 42: Pass@1 = 0.3600, Cell Acc = 0.8822
Evaluating structured pruning for ARC 2024 (ratio=0.25)...
  Seed 42: Pass@1 = 0.0000, Cell Acc = 0.0291, Latency = 20.63 ms
  Seed 42: Pass@1 = 0.0000, Cell Acc = 0.0291
Evaluating structured pruning for ARC 2024 (ratio=0.5)...
  Seed 42: Pass@1 = 0.0000, Cell Acc = 0.0000, Latency = 20.48 ms
  Seed 42: Pass@1 = 0.0000, Cell Acc = 0.0000
Evaluating structured pruning for Maze Hard (ratio=0.0)...
  Seed 42: Pass@1 = 0.8680, Cell Acc = 0.9952, Latency = 19.25 ms
  Seed 42: Pass@1 = 0.8680, Cell Acc = 0.9952
Evaluating structured pruning for Maze Hard (ratio=0.25)...
  Seed 42: Pass@1 = 0.0000, Cell Acc = 0.8728, Latency = 19.60 ms
  Seed 42: Pass@1 = 0.0000, Cell Acc = 0.8728
Evaluating structured pruning for Maze Hard (ratio=0.5)...
  Seed 42: Pass@1 = 0.0000, Cell Acc = 0.1569, Latency = 19.51 ms
  Seed 42: Pass@1 = 0

Seed                           42
Model          Prune Ratio       
ARC 2024       0.00         0.360
               0.25         0.000
               0.50         0.000
Maze Hard      0.00         0.868
               0.25         0.000
               0.50         0.000
Sudoku Extreme 0.00         0.691
               0.25         0.000
               0.50         0.000

## Section 14 — TorchScript Export (Edge-Native Serialization)

In [39]:
class TRMBackboneStep(nn.Module):
    """Single H-cycle backbone step. Puzzle embedding passed as float input."""
    def __init__(self, inner_model):
        super().__init__()
        self.m = inner_model

    def forward(self, x, puzzle_emb_row, z_H, z_L):
        from models.recursive_reasoning.trm import (
            TinyRecursiveReasoningModel_ACTV1Carry,
            TinyRecursiveReasoningModel_ACTV1InnerCarry,
        )
        # Patch puzzle_emb to return the pre-looked-up row
        orig = self.m.inner.puzzle_emb.forward
        cast_to = self.m.inner.puzzle_emb.cast_to
        def _injected(ids): return puzzle_emb_row.to(cast_to)
        self.m.inner.puzzle_emb.forward = _injected

        pids  = torch.zeros(x.shape[0], dtype=torch.int32, device=x.device)
        batch = {"inputs": x.to(torch.int32), "labels": x.to(torch.int32),
                 "puzzle_identifiers": pids}
        carry = TinyRecursiveReasoningModel_ACTV1Carry(
            inner_carry=TinyRecursiveReasoningModel_ACTV1InnerCarry(z_H=z_H, z_L=z_L),
            steps=torch.zeros(x.shape[0], dtype=torch.int32, device=x.device),
            halted=torch.zeros(x.shape[0], dtype=torch.bool, device=x.device),
            current_data=batch,
        )
        try:
            new_carry, outputs = self.m(carry, batch)
        finally:
            self.m.inner.puzzle_emb.forward = orig
        return outputs["logits"], new_carry.inner_carry.z_H, new_carry.inner_carry.z_L

trace_records = []
for model_name, model_obj, loader, _ in models_list:
    if loader is None: continue
    inner = get_inner(model_obj)
    
    dev     = str(DEVICE)
    seq_len = inner.config.seq_len
    emb_len = inner.inner.puzzle_emb_len
    hidden  = inner.config.hidden_size
    emb_dim = inner.inner.puzzle_emb.weights.shape[1]

    wrapper = TRMBackboneStep(inner).eval().to(dev)
    wrapper = wrapper.float()

    # Example inputs on the same device as the model
    x_d   = torch.zeros(1, seq_len,              dtype=torch.int64,  device=dev)
    emb_d = torch.zeros(1, emb_dim,              dtype=torch.float32, device=dev)
    zH_d  = torch.zeros(1, seq_len + emb_len, hidden, dtype=torch.float32, device=dev)
    zL_d  = torch.zeros(1, seq_len + emb_len, hidden, dtype=torch.float32, device=dev)
    
    t0 = time.time()
    try:
        with torch.no_grad():
            traced = torch.jit.trace(wrapper, (x_d, emb_d, zH_d, zL_d), strict=False)
        trace_time = time.time() - t0
        success = True
    except Exception as e:
        print(f"Tracing failed for {model_name}: {e}")
        trace_time = 0.0
        success = False
        
    trace_records.append({
        "Model": model_name, "Tracing Success": success, "Trace Time (s)": trace_time
    })
df_trace = pd.DataFrame(trace_records)
df_trace

,Model,Tracing Success,Trace Time (s)
0,ARC 2024,True,2.201543
1,Maze Hard,True,2.014039
2,Sudoku Extreme,True,0.957831


## Section 15 — QAT with Proper Train / Val Split

In [ ]:
# Validated QAT execution commented out by default
# qat_val_records = []
# for model_name, model_obj, loader, n_sup in models_list:
#     if loader is None: continue
#     for seed in [42, 43, 44]:
#         m = copy.deepcopy(model_obj)
#         loss = run_qat_loop(m, loader, steps=3, micro_batch_size=64)
#         metrics = evaluate_across_seeds(model_name, m, loader, seeds=[seed], n_sup_max=n_sup)[0]
#         qat_val_records.append({
#             "Model": model_name, "Seed": seed, "QAT Val Loss": loss, "QAT Val Pass@1": metrics["exact1"]
#         })
# df_qat_val = pd.DataFrame(qat_val_records)
# df_qat_val

## Section 16 — INT8 Backbone + Single-Puzzle Fused Artifact

In [40]:
fused_records = []
for model_name, model_obj, loader, n_sup in models_list:
    if loader is None: continue
    print(f"Evaluating Fused INT8 + Single-Puzzle for {model_name}...")
    m_int8 = quantize_int8_bnb(model_obj)
    inner = get_inner(m_int8)
    active_row = inner.puzzle_emb(torch.tensor([0], device=DEVICE))
    inner.puzzle_emb = SinglePuzzleEmbedding(active_row)
    res = evaluate_across_seeds(model_name, m_int8, loader, n_sup_max=n_sup)
    for r in res:
        print(f"  Seed {r['seed']}: Pass@1 = {r['exact1']:.4f}, Cell Acc = {r['cell']:.4f}")
        fused_records.append({
            "Model": model_name, "Seed": r["seed"], "Fused INT8+Single-Puzzle Pass@1": r["exact1"], "Cell Acc": r["cell"]
        })
df_fused = pd.DataFrame(fused_records)
df_fused

Evaluating Fused INT8 + Single-Puzzle for ARC 2024...
  Seed 42: Pass@1 = 0.3600, Cell Acc = 0.8827, Latency = 75.54 ms
  Seed 42: Pass@1 = 0.3600, Cell Acc = 0.8827
Evaluating Fused INT8 + Single-Puzzle for Maze Hard...
  Seed 42: Pass@1 = 0.8700, Cell Acc = 0.9953, Latency = 61.10 ms
  Seed 42: Pass@1 = 0.8700, Cell Acc = 0.9953
Evaluating Fused INT8 + Single-Puzzle for Sudoku Extreme...
  Seed 42: Pass@1 = 0.6900, Cell Acc = 0.8749, Latency = 8.48 ms
  Seed 42: Pass@1 = 0.6900, Cell Acc = 0.8749


,Model,Seed,Fused INT8+Single-Puzzle Pass@1,Cell Acc
0,ARC 2024,42,0.36,0.882749
1,Maze Hard,42,0.87,0.995291
2,Sudoku Extreme,42,0.69,0.874938


## Section 17 — Simulated Edge Deployment Profile

In [42]:
tradeoffs = [
    {"Model": "ARC 2024", "model_obj": model_arc, "loader": loader_arc, "H": 1, "n_sup": 8},
    {"Model": "Maze Hard", "model_obj": model_maze, "loader": loader_maze, "H": 1, "n_sup": 8},
    {"Model": "Sudoku Extreme", "model_obj": model_sudoku, "loader": loader_sudoku, "H": 3, "n_sup": 10}
]
profile_records = []
for item in tradeoffs:
    name = item["Model"]
    mdl = item["model_obj"]
    loader = item["loader"]
    H = item["H"]
    n_sup = item["n_sup"]
    if loader is None: continue
    
    print(f"Profiling {name} (H={H}, n_sup={n_sup})...")
    q_mdl = quantize_int8_bnb(mdl)
    inner = get_inner(q_mdl)
    orig_H = inner.config.H_cycles
    inner.config.H_cycles = H
    
    batch = next(iter(loader))
    inputs, labels, pids = [t.to(DEVICE) for t in batch]
    batch_dict = {"inputs": inputs.to(torch.int32), "labels": labels.to(torch.int32), "puzzle_identifiers": pids.to(torch.int32)}
    
    from torch.profiler import profile, ProfilerActivity
    with profile(activities=[ProfilerActivity.CPU], record_shapes=True, with_flops=True) as prof:
        carry = inner.initial_carry(batch_dict)
        from models.recursive_reasoning.trm import TinyRecursiveReasoningModel_ACTV1Carry, TinyRecursiveReasoningModel_ACTV1InnerCarry
        ic = carry.inner_carry
        carry = TinyRecursiveReasoningModel_ACTV1Carry(
            inner_carry=TinyRecursiveReasoningModel_ACTV1InnerCarry(z_H=ic.z_H.to(DEVICE), z_L=ic.z_L.to(DEVICE)),
            steps=carry.steps.to(DEVICE),
            halted=carry.halted.to(DEVICE),
            current_data={k: v.to(DEVICE) for k, v in carry.current_data.items()},
        )
        for _ in range(n_sup):
            carry, _ = inner(carry, batch_dict)
            
    total_flops = sum(event.flops for event in prof.key_averages() if event.flops is not None)
    gflops = (total_flops / 1e9) / inputs.shape[0]
    
    inner.config.H_cycles = orig_H
    res = evaluate_across_seeds(name, q_mdl, loader, n_sup_max=n_sup)
    for r in res:
        print(f"  Seed {r['seed']}: GFLOPs = {gflops:.4f}, Pass@1 = {r['exact1']:.4f}, Latency = {r['ms']:.2f} ms")
        profile_records.append({
            "Model": name, "Seed": r["seed"], "GFLOPs": gflops, "Pass@1": r["exact1"], "Latency (ms)": r["ms"]
        })
df_edge = pd.DataFrame(profile_records)
df_edge

Profiling ARC 2024 (H=1, n_sup=8)...
  Seed 42: Pass@1 = 0.3600, Cell Acc = 0.8755, Latency = 60.69 ms
  Seed 42: GFLOPs = 34.1115, Pass@1 = 0.3600, Latency = 60.69 ms
Profiling Maze Hard (H=1, n_sup=8)...
  Seed 42: Pass@1 = 0.8700, Cell Acc = 0.9953, Latency = 49.20 ms
  Seed 42: GFLOPs = 14.8618, Pass@1 = 0.8700, Latency = 49.20 ms
Profiling Sudoku Extreme (H=3, n_sup=10)...
  Seed 42: Pass@1 = 0.6900, Cell Acc = 0.8749, Latency = 8.51 ms
  Seed 42: GFLOPs = 102.5377, Pass@1 = 0.6900, Latency = 8.51 ms


,Model,Seed,GFLOPs,Pass@1,Latency (ms)
0,ARC 2024,42,34.111460,0.36,60.690460
1,Maze Hard,42,14.861825,0.87,49.204230
2,Sudoku Extreme,42,102.537682,0.69,8.510652


## Section 18 — Pruning, Distillation, and Attention Swaps

In [ ]:
student_config = copy.deepcopy(arc_config)
student_config["L_layers"] = 1

prun_dist_records = []
for model_name, model_obj, loader, n_sup in models_list:
    if loader is None: continue
    for seed in [42, 43, 44]:
        print(f"Evaluating student/pruned comparison for {model_name} (Seed {seed})...")
        torch.manual_seed(seed)
        p_model = prune_structured(model_obj, amount=0.25)
        student = TinyRecursiveReasoningModel_ACTV1(config_dict=student_config).to(DEVICE)
        m_pruned = evaluate_across_seeds(model_name, p_model, loader, seeds=[seed], n_sup_max=n_sup)[0]
        m_stud = evaluate_across_seeds(model_name, student, loader, seeds=[seed], n_sup_max=n_sup)[0]
        
        print(f"  Pruned Pass@1 = {m_pruned['exact1']:.4f}, Student Pass@1 = {m_stud['exact1']:.4f}")
        prun_dist_records.append({
            "Model": model_name, "Seed": seed, "Pruned Pass@1": m_pruned["exact1"], "Student Pass@1": m_stud["exact1"]
        })
df_prun_dist = pd.DataFrame(prun_dist_records)
df_prun_dist

## Section 19 — Sequence Length Ablation

In [43]:
ablation_records = []
for model_name, model_obj, loader, n_sup in models_list:
    if loader is None: continue
    lens = [900, 500, 200] if "sudoku" not in model_name.lower() else [81, 40, 20]
    for length in lens:
        print(f"Evaluating {model_name} with context length truncated to {length}...")
        res = evaluate_across_seeds(model_name, model_obj, loader, n_sup_max=n_sup, trunc_len=length)
        for r in res:
            print(f"  Seed {r['seed']}: Pass@1 = {r['exact1']:.4f}, Cell Acc = {r['cell']:.4f}")
            ablation_records.append({
                "Model": model_name, "Context Len": length, "Seed": r["seed"], "Pass@1": r["exact1"], "Cell Acc": r["cell"]
            })
df_ablate = pd.DataFrame(ablation_records)
df_ablate.pivot(index=["Model", "Context Len"], columns="Seed", values="Pass@1")

Evaluating ARC 2024 with context length truncated to 900...
  Seed 42: Pass@1 = 0.3600, Cell Acc = 0.8755, Latency = 71.07 ms
  Seed 42: Pass@1 = 0.3600, Cell Acc = 0.8755
Evaluating ARC 2024 with context length truncated to 500...
  Seed 42: Pass@1 = 0.2950, Cell Acc = 0.3274, Latency = 71.07 ms
  Seed 42: Pass@1 = 0.2950, Cell Acc = 0.3274
Evaluating ARC 2024 with context length truncated to 200...
  Seed 42: Pass@1 = 0.1025, Cell Acc = 0.0580, Latency = 71.36 ms
  Seed 42: Pass@1 = 0.1025, Cell Acc = 0.0580
Evaluating Maze Hard with context length truncated to 900...
  Seed 42: Pass@1 = 0.8700, Cell Acc = 0.9953, Latency = 68.12 ms
  Seed 42: Pass@1 = 0.8700, Cell Acc = 0.9953
Evaluating Maze Hard with context length truncated to 500...
  Seed 42: Pass@1 = 0.0000, Cell Acc = 0.7001, Latency = 68.02 ms
  Seed 42: Pass@1 = 0.0000, Cell Acc = 0.7001
Evaluating Maze Hard with context length truncated to 200...
  Seed 42: Pass@1 = 0.0000, Cell Acc = 0.4998, Latency = 68.10 ms
  Seed 42: 

Seed                            42
Model          Context Len        
ARC 2024       200          0.1025
               500          0.2950
               900          0.3600
Maze Hard      200          0.0000
               500          0.0000
               900          0.8700
Sudoku Extreme 20           0.0000
               40           0.0000
               81           0.6830

## Section 20 — Better QAT: Starting from Calibrated INT4

In [ ]:
# Better QAT execution commented out by default
# better_qat_records = []
# for model_name, model_obj, loader, n_sup in models_list:
#     if loader is None: continue
#     for seed in [42, 43, 44]:
#         torch.manual_seed(seed)
#         m_cal = quantize_calibrated_int4(model_obj)
#         loss = run_qat_loop(m_cal, loader, steps=5, micro_batch_size=64)
#         metrics = evaluate_across_seeds(model_name, m_cal, loader, seeds=[seed], n_sup_max=n_sup)[0]
#         better_qat_records.append({
#             "Model": model_name, "Seed": seed, "Loss": loss, "QAT-Cal Pass@1": metrics["exact1"]
#         })
# df_better = pd.DataFrame(better_qat_records)
# df_better

## Section 21 — Knowledge Distillation: Training a Smaller Student

In [44]:
def run_distillation_loop(teacher, student, loader, steps=5, micro_batch_size=64):
    teacher.eval()
    student.train()
    optimizer = torch.optim.Adam(student.parameters(), lr=1e-5)
    
    batch = next(iter(loader))
    inputs, labels, pids = [t.to(DEVICE) for t in batch]
    num_micro_batches = max(1, len(inputs) // micro_batch_size)
    
    losses = []
    for _ in range(steps):
        optimizer.zero_grad()
        mb_loss = 0.0
        for mb_idx in range(num_micro_batches):
            mb_start = mb_idx * micro_batch_size
            mb_end = mb_start + micro_batch_size
            mb_x = inputs[mb_start:mb_end]
            mb_y = labels[mb_start:mb_end]
            mb_pids = pids[mb_start:mb_end]
            
            batch_dict = {"inputs": mb_x.to(torch.int32), "labels": mb_y.to(torch.int32), "puzzle_identifiers": mb_pids.to(torch.int32)}
            with torch.no_grad():
                carry_t = teacher.initial_carry(batch_dict)
                _, out_t = teacher(carry_t, batch_dict)
                
            carry_s = student.initial_carry(batch_dict)
            _, out_s = student(carry_s, batch_dict)
            loss = F.kl_div(
                F.log_softmax(out_s["logits"], dim=-1),
                F.softmax(out_t["logits"], dim=-1),
                reduction="batchmean"
            )
            mb_loss = mb_loss + loss
            
        mb_loss_normalized = mb_loss / num_micro_batches
        mb_loss_normalized.backward()
        optimizer.step()
        losses.append(mb_loss_normalized.item())
    return np.mean(losses)

# Knowledge Distillation training execution commented out by default
# distill_records = []
# for model_name, model_obj, loader, _ in models_list:
#     if loader is None: continue
#     for seed in [42, 43, 44]:
#         torch.manual_seed(seed)
#         student = TinyRecursiveReasoningModel_ACTV1(config_dict=student_config).to(DEVICE)
#         loss = run_distillation_loop(model_obj, student, loader, steps=5, micro_batch_size=64)
#         distill_records.append({
#             "Model": model_name, "Seed": seed, "Distill Loss": loss
#         })
# df_distill = pd.DataFrame(distill_records)
# df_distill

## Section 22 — Linear Attention Approximation

In [45]:
class LinearAttentionApproximation(nn.Module):
    def __init__(self, original_attn):
        super().__init__()
        self.q_proj = original_attn.qkv_proj
    def forward(self, x):
        return x

def apply_linear_attn_approx(mdl):
    m = copy.deepcopy(get_inner(mdl))
    for name, module in list(m.named_modules()):
        if hasattr(module, "self_attn") and module.self_attn is not None:
            module.self_attn = LinearAttentionApproximation(module.self_attn)
    return m

# attn_records = []
# for model_name, model_obj, loader, n_sup in models_list:
#     if loader is None: continue
#     if "sudoku" in model_name.lower():
#         print(f"{model_name} uses MLP mixer block. Skipping attention swaps.")
#         continue
#     print(f"Evaluating Linear Attention Approximation for {model_name}...")
#     m_approx = apply_linear_attn_approx(model_obj)
#     res = evaluate_across_seeds(model_name, m_approx, loader, n_sup_max=n_sup)
#     for r in res:
#         print(f"  Seed {r['seed']}: Pass@1 = {r['exact1']:.4f}, Cell Acc = {r['cell']:.4f}")
#         attn_records.append({
#             "Model": model_name, "Seed": r["seed"], "Linear Attn Pass@1": r["exact1"], "Cell Acc": r["cell"]
#         })
# df_attn = pd.DataFrame(attn_records)
# df_attn